In [1]:
import torch
from tqdm.auto import tqdm # Use tqdm.auto for compatibility in notebooks/scripts
import numpy as np

class A2CTrainer:
    def __init__(self, agent, dataloader, num_epochs=3, gradient_accumulation_steps=1, log_interval=10, device='cuda'):
        """
        Initializes the A2CTrainer for an A2C agent (e.g., SynthesizerAgent).

        :param agent: The A2C agent (e.g., SynthesizerAgent) to train.
        :param dataloader: DataLoader providing tokenized batches (input_ids, attention_mask, labels).
        :param num_epochs: Number of training epochs.
        :param gradient_accumulation_steps: Number of batches to accumulate gradients over before an optimizer step.
        :param log_interval: How often (in batches) to print/log current progress.
        :param device: Device to use (default: 'cuda').
        """
        self.agent = agent
        self.dataloader = dataloader
        self.num_epochs = num_epochs
        self.gradient_accumulation_steps = gradient_accumulation_steps
        self.log_interval = log_interval
        self.device = device

        # Ensuring agent is on the correct device
        self.agent.to(self.device)

    def train(self):
        self.agent.train() # Set agent to training mode

        all_epoch_avg_rewards = []
        global_step = 0 # Tracks total batches processed

        for epoch in range(self.num_epochs):
            total_epoch_policy_loss = 0
            total_epoch_value_loss = 0
            total_epoch_reward = 0
            batches_processed_this_epoch = 0

            print(f"\nEpoch {epoch + 1}/{self.num_epochs}")

            # Reset optimizer gradients at the start of each epoch
            # This is done *before* the inner loop to ensure a fresh start for accumulation
            self.agent.optimizer.zero_grad()

            # Use tqdm for a progress bar over the dataloader
            progress_bar = tqdm(self.dataloader, desc=f"Epoch {epoch+1} Training")

            for step, batch in enumerate(progress_bar):
                # Move batch tensors to the correct device
                input_ids = batch['input_ids'].to(self.device)
                attention_mask = batch['attention_mask'].to(self.device)
                labels = batch['labels'].to(self.device) # Tokenized reference summaries

                try:
                    # Call the batched training step method on the agent
                    # This method performs forward pass and backward pass, but doesn't step optimizer or zero_grad
                    _, rewards_batch, policy_loss_batch, value_loss_batch, _ = self.agent.train_step_batched(
                        input_ids=input_ids,
                        attention_mask=attention_mask,
                        labels=labels
                    )

                    total_epoch_policy_loss += policy_loss_batch.item()
                    total_epoch_value_loss += value_loss_batch.item()
                    total_epoch_reward += rewards_batch.mean().item() # Mean reward for the batch
                    batches_processed_this_epoch += 1
                    global_step += 1

                    # Perform optimizer step if gradient accumulation criteria met
                    if (global_step % self.gradient_accumulation_steps == 0):
                        self.agent.optimizer.step()
                        self.agent.optimizer.zero_grad() # Clear gradients for the next accumulation cycle

                        # Log progress after each actual optimizer step
                        if global_step % self.log_interval == 0:
                            # Calculate average over current processed batches this epoch for logging
                            # Use total accumulated values divided by the number of batches *that have been accumulated over*
                            current_avg_p_loss = total_epoch_policy_loss / batches_processed_this_epoch
                            current_avg_v_loss = total_epoch_value_loss / batches_processed_this_epoch
                            current_avg_reward = total_epoch_reward / batches_processed_this_epoch

                            progress_bar.set_postfix(
                                {
                                    "Avg_Rew": f"{current_avg_reward:.4f}",
                                    "Avg_P_Loss": f"{current_avg_p_loss:.4f}",
                                    "Avg_V_Loss": f"{current_avg_v_loss:.4f}"
                                }
                            )

                except RuntimeError as e:
                    print(f"\n Runtime error at Step {step + 1}: {e}. Attempting to clear cache and skip batch.")
                    if "out of memory" in str(e):
                        torch.cuda.empty_cache() # Try to free memory
                    self.agent.optimizer.zero_grad() # Clear any partial gradients that might have accumulated
                    continue # Skip to next batch
                except Exception as e:
                    print(f"\n General error at Step {step + 1}: {e}. Skipping batch.")
                    self.agent.optimizer.zero_grad() # Clear any partial gradients
                    continue # Skip to next batch

            # Perform a final optimizer step for any remaining accumulated gradients at epoch end
            # This handles cases where total batches are not a multiple of gradient_accumulation_steps
            if (global_step % self.gradient_accumulation_steps != 0) and (batches_processed_this_epoch > 0):
                self.agent.optimizer.step()
                self.agent.optimizer.zero_grad()

            # --- Epoch Summary ---
            if batches_processed_this_epoch > 0:
                avg_epoch_reward = total_epoch_reward / batches_processed_this_epoch
                avg_epoch_policy_loss = total_epoch_policy_loss / batches_processed_this_epoch
                avg_epoch_value_loss = total_epoch_value_loss / batches_processed_this_epoch
                all_epoch_avg_rewards.append(avg_epoch_reward)

                print(f"Epoch {epoch + 1} Complete - "
                      f"Avg Reward: {avg_epoch_reward:.4f} | "
                      f"Avg Policy Loss: {avg_epoch_policy_loss:.4f} | "
                      f"Avg Value Loss: {avg_epoch_value_loss:.4f}")
            else:
                print(f"Epoch {epoch + 1} Complete - No batches processed in this epoch.")

        print("\n--- Training Finished ---")
        return all_epoch_avg_rewards